In [ ]:
from datasets import ClassLabel, Dataset, Sequence, concatenate_datasets, load_dataset, Features, Image, Value, DatasetDict, load_from_disk
from joblib import Parallel, delayed
from tqdm import tqdm

from datasets import concatenate_datasets
import numpy as np
import os

In [ ]:
num_proc = min(32, os.cpu_count())
dataset = load_dataset("project-oceania/planktonzilla_full", split="train")

In [ ]:
def clean_corrupt_examples_optimized(dataset, batch_size=100, n_jobs=-1):
    """
    Elimina ejemplos corruptos usando lectura por lotes para maximizar velocidad.
    """
    total_size = len(dataset)
    
    # Función que procesa un bloque de índices
    def process_batch(start_idx):
        end_idx = min(start_idx + batch_size, total_size)
        batch_range = range(start_idx, end_idx)
        
        try:
            # --- 1. Intento Optimista ---
            # Intentamos acceder al bloque entero. 
            # Si HF puede leer este slice sin error, todos están sanos.
            _ = dataset[start_idx:end_idx] 
            return list(batch_range) # Retornamos todos los índices del lote
            
        except Exception:
            # --- 2. Fallback Pesimista ---
            # Si el bloque falla, iteramos uno por uno SOLO en este bloque
            valid_indices = []
            for i in batch_range:
                try:
                    _ = dataset[i]
                    valid_indices.append(i)
                except Exception:
                    # Aquí encontramos el archivo corrupto y simplemente lo ignoramos
                    pass
            return valid_indices

    # Generamos los puntos de inicio para los batches
    starts = range(0, total_size, batch_size)

    # Paralelizamos por BATCH, no por fila (reduciendo overhead masivamente)
    results = Parallel(n_jobs=n_jobs)(
        delayed(process_batch)(s) for s in tqdm(starts, desc="Verificando integridad")
    )

    # Aplanamos la lista de listas resultante
    good_indices = [idx for batch in results for idx in batch]

    print(f"Original: {total_size} -> Limpio: {len(good_indices)}")
    print(f"Eliminados: {total_size - len(good_indices)} ejemplos corruptos.")

    return dataset.select(good_indices)

## Creating planktonzilla only plankton

In [ ]:
def build_final_planktonzilla(ds, num_proc=1):


    TAXONOMY_COLS = [
    "Kingdom", "Phylum", "Class",
    "Order", "Family", "Genus", "Species"
    ]


    # delete corrupt examples
    ds = clean_corrupt_examples_optimized(ds, batch_size=1000, n_jobs=-1)
    
    # select only plankton examples with taxo
    ds = ds.filter(lambda x: x["Kingdom"] != "", num_proc=num_proc)
    ds = ds.filter(lambda x: x["plankton"] == True, num_proc=num_proc)

    # ----- construir string taxonómico -----
    def build_tax_string(example):
        tax = [example[c] for c in TAXONOMY_COLS if example[c] not in ("", None)]
        return {"tax_label": " ".join(tax)}

    ds = ds.map(build_tax_string, num_proc=num_proc)

    # ----- obtener clases únicas -----
    unique_labels = sorted(set(ds["tax_label"]))
    class_label = ClassLabel(names=unique_labels)

    # ----- codificar labels -----
    def encode_label(example):
        return {"label": class_label.str2int(example["tax_label"])}

    ds = ds.map(encode_label, num_proc=num_proc)

    # ----- dejar solo image + label + dataset -----
    ds = ds.remove_columns(
        [c for c in ds.column_names if c not in ["image", "label", "dataset"]]
    )

    new_features = Features({
        "image": ds.features["image"],
        "label": class_label,
        "dataset": Value("string")
    })

    ds = ds.cast(new_features)

    return ds


In [ ]:
dataset_tax = build_final_planktonzilla(
    dataset,
    num_proc=num_proc
)

In [ ]:
from datasets import ClassLabel, Value
old_classlabel = dataset_tax.features["label"]
old_names = old_classlabel.names


replacements = {
    "animalia cnidaria hydrozoa leptothecata cylindrotheca":
        "chromista heterokontophyta bacillariophyceae bacillariales bacillariaceae cylindrotheca",

    "animalia mollusca bivalvia inoceramidae cladoceramus":
        "animalia mollusca bivalvia pteriida inoceramidae cladoceramus",

    "chromista heterokontophyta bacillariophyceae rhizosoleniales rhizosoleniaceae rhizosolenia richelia":
        "chromista heterokontophyta bacillariophyceae rhizosoleniales rhizosoleniaceae rhizosolenia",
}

In [ ]:
canonical_names = []
for name in old_names:
    canonical_names.append(replacements.get(name, name))


In [ ]:
def remap_label(ex):
    ex["label"] = old_id_to_new_id[ex["label"]]
    return ex

dataset_tax = dataset_tax.map(remap_label, num_proc=32)


In [ ]:
dataset_tax = dataset_tax.cast_column(
    "label",
    ClassLabel(names=new_names)
)


In [ ]:
from collections import Counter
import numpy as np
from datasets import concatenate_datasets


def stratified_split_by_dataset(
    ds,
    num_proc,
    seed=42,
    test_split_value=0.2,
    val_split_value=0.2,
):
    train_splits = []
    val_splits = []
    test_splits = []

    dataset_names = sorted(set(ds["dataset"]))

    for dname in dataset_names:
        ds_sub = ds.filter(
            lambda x: x["dataset"] == dname,
            num_proc=num_proc
        )

        labels = ds_sub["label"]
        counts = Counter(labels)

        # ---- clases con 1 solo ejemplo → forzar a train
        singleton_labels = {k for k, v in counts.items() if v == 1}

        singleton_idx = [
            i for i, y in enumerate(labels) if y in singleton_labels
        ]

        remaining_idx = [
            i for i in range(len(ds_sub)) if i not in singleton_idx
        ]

        ds_singleton = ds_sub.select(singleton_idx) if singleton_idx else None
        ds_remaining = ds_sub.select(remaining_idx) if remaining_idx else None

        # Si no queda nada para splittear, todo va a train
        if ds_remaining is None or len(ds_remaining) == 0:
            train_splits.append(ds_sub)
            continue

        n_samples = len(ds_remaining)

        # -------- train / (val+test) --------
        try:
            splits = ds_remaining.train_test_split(
                test_size=int(n_samples * (test_split_value + val_split_value)),
                shuffle=True,
                seed=seed,
                stratify_by_column="label",
            )
        except ValueError:
            splits = ds_remaining.train_test_split(
                test_size=int(n_samples * (test_split_value + val_split_value)),
                shuffle=True,
                seed=seed,
            )

        train_split = splits["train"]
        val_test_split = splits["test"]

        # -------- val / test --------
        try:
            splits = val_test_split.train_test_split(
                test_size=int(n_samples * val_split_value),
                shuffle=True,
                seed=seed,
                stratify_by_column="label",
            )
        except ValueError:
            splits = val_test_split.train_test_split(
                test_size=int(n_samples * val_split_value),
                shuffle=True,
                seed=seed,
            )

        test_split = splits["train"]
        val_split = splits["test"]

        # ---- añadir singletons forzados a train
        if ds_singleton is not None:
            train_split = concatenate_datasets([train_split, ds_singleton])

        train_splits.append(train_split)
        val_splits.append(val_split)
        test_splits.append(test_split)

    # -------- concatenar splits finales --------
    train_ds = concatenate_datasets(train_splits)
    val_ds   = concatenate_datasets(val_splits) if val_splits else None
    test_ds  = concatenate_datasets(test_splits) if test_splits else None

    return train_ds, val_ds, test_ds


In [ ]:
train_ds, val_ds, test_ds = stratified_split_by_dataset(dataset, num_proc=num_proc)

In [ ]:
hf_dataset = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
    "test": test_ds
})

In [ ]:
hf_dataset.save_to_disk(f"/lustre/fsn1/projects/rech/tec/uod68bo/data/planktonzilla_only_plankton")

## Generating planktonzilla only OOD

In [ ]:
def build_ood(ds, num_proc=1):

    # 1️⃣ eliminar ejemplos corruptos
    ds = clean_corrupt_examples_optimized(ds, batch_size=1000, n_jobs=-1)

    # 2️⃣ filtrar solo los que NO son plankton
    ds = ds.filter(lambda x: x["plankton"] == False, num_proc=num_proc)

    # 3️⃣ obtener clases únicas desde original_label
    unique_labels = sorted(set(ds["original_label"]))
    class_label = ClassLabel(names=unique_labels)

    # 4️⃣ codificar original_label → label
    def encode_label(example):
        return {
            "label": class_label.str2int(example["original_label"])
        }

    ds = ds.map(encode_label, num_proc=num_proc)

    # 5️⃣ dejar solo columnas necesarias
    ds = ds.remove_columns(
        [c for c in ds.column_names if c not in ["image", "label", "dataset"]]
    )

    # 6️⃣ redefinir features
    new_features = Features({
        "image": ds.features["image"],
        "label": class_label,
        "dataset": Value("string")
    })

    ds = ds.cast(new_features)

    return ds

In [ ]:
dataset_ood = build_ood(
    dataset,
    num_proc=num_proc
)

In [ ]:
hf_dataset = DatasetDict({
    "train": dataset_ood,
})

In [ ]:
hf_dataset.save_to_disk(f"/lustre/fsn1/projects/rech/tec/uod68bo/data/planktonzilla_ood")